# ICP For Point-Cloud Registration

Implementing the ICP algorithm to register two point clouds. 
- Registration refers to aligning to be coherent. 
- In robotics, when lidar scans from different parts of a room, for instance, need to be joined together to create a full map of the environment.
- Some examples in [this paper](http://redwood-data.org/indoor_lidar_rgbd/paper.pdf).

## Setup 

- This notebook originally from [CS237a hw3, part3](https://colab.research.google.com/drive/1O94OB54oYSu7E1CAxP29IeuCnfJnA3t8?usp=sharing)
- Use the [Stanford Bunny](https://graphics.stanford.edu/data/3Dscanrep/) dataset
- Referring to partial point cloud, $A$, as the "source", 
- Referring to full scan, $B$, of the bunny as the "target"

```python
cloud_A = o3d.io.read_point_cloud("data/bun045.ply")
cloud_B = o3d.io.read_point_cloud("data/bun_zipper.ply")
```

In [1]:
# In terminal, create a virtual env, and install dependencies
#
# uv venv
# source .venv/bin/activate
# uv pip install open3d 
# uv pip install ipykernel
# uv pip install ipywidgets
# code .
#  
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from utils import *

cloud_A = o3d.io.read_point_cloud("data/bun045.ply")
cloud_B = o3d.io.read_point_cloud("data/bun_zipper.ply")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Visualize point cloud data

- Use the slider bars to view the bunny from different angles.
- Partial point cloud $A$ scan is not a complete scan of the bunny
- We will be referring to this partial point cloud, $A$, as the "source", 
- We will be referring to the full scan $B_t$ of the bunny as the "target"

In [2]:
widgets.interact(
    plot_point_clouds,
    pcd_list=widgets.fixed([cloud_A]),
    azim=widgets.IntSlider(-90, min=-180, max=180, step=5, description="Azimuth"),
    elev=widgets.IntSlider(90, min=0, max=90, step=5, description="Elevation")
)

interactive(children=(IntSlider(value=-90, description='Azimuth', max=180, min=-180, step=5), IntSlider(value=…

<function utils.plot_point_clouds(pcd_list, azim=-60, elev=30)>

In [3]:
widgets.interact(
    plot_point_clouds,
    pcd_list=widgets.fixed([cloud_B]),
    azim=widgets.IntSlider(-90, min=-180, max=180, step=5, description="Azimuth"),
    elev=widgets.IntSlider(90, min=0, max=90, step=5, description="Elevation")
)

interactive(children=(IntSlider(value=-90, description='Azimuth', max=180, min=-180, step=5), IntSlider(value=…

<function utils.plot_point_clouds(pcd_list, azim=-60, elev=30)>

In [4]:
# View both clouds A (partial_scan) and B (full_bunny)
widgets.interact(
    plot_point_clouds,
    pcd_list=widgets.fixed([cloud_A, cloud_B]),
    azim=widgets.IntSlider(-90, min=-180, max=180, step=5, description="Azimuth"),
    elev=widgets.IntSlider(90, min=0, max=90, step=5, description="Elevation")
)

interactive(children=(IntSlider(value=-90, description='Azimuth', max=180, min=-180, step=5), IntSlider(value=…

<function utils.plot_point_clouds(pcd_list, azim=-60, elev=30)>

# Global Registration

Our goal will be to match the point cloud data
- containing a partial view of the bunny (A)
- with the full 3D mesh model of the same bunny (B)

We will do this using the RANSAC algorithm.


$$ 
\text{Given A, B} \\
\text{find a matrix} \quad T \\
\text{s.t.} \quad B = A T 
$$

## Extract features and downsample
We can see that the two point clouds are clearly misaligned right now. 

- So first, we will extract some features of each point cloud. 
- These features, called the "FPFH" features of the point clouds are a vector of 33 values for each point in the point cloud that represents some unique features of that point. 
- Therefore, if we have N points in point cloud, your FPFH feature matrix for that point cloud will be of shape (N, 33).

Since N can be large for raw point cloud data, we will downsample it a bit so that it is easier to experiment with.

In [5]:
# Extract features
def preprocess_point_cloud(point_cloud, voxel_size):
    print(":: Downsample with a voxel size %.3f." % voxel_size)
    point_cloud_sample = point_cloud.voxel_down_sample(voxel_size)

    radius_normal = voxel_size * 2
    print(":: Estimate normal with search radius %.3f." % radius_normal)
    point_cloud_sample.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=30))

    radius_feature = voxel_size * 5
    print(":: Compute FPFH feature with search radius %.3f." % radius_feature)
    point_cloud_features = o3d.pipelines.registration.compute_fpfh_feature(
        point_cloud_sample,
        o3d.geometry.KDTreeSearchParamHybrid(radius=radius_feature, max_nn=100))
    
    return point_cloud_sample, point_cloud_features

def prepare_dataset(cloud_A, cloud_B, voxel_size):
    A_sample, A_features = preprocess_point_cloud(cloud_A, voxel_size)
    B_sample, B_features = preprocess_point_cloud(cloud_B, voxel_size)
    return cloud_A, cloud_B, A_sample, B_sample, A_features, B_features

cloud_A, cloud_B, A_sample, B_sample, A_features, B_features = prepare_dataset(cloud_A, cloud_B, 0.01)

:: Downsample with a voxel size 0.010.
:: Estimate normal with search radius 0.020.
:: Compute FPFH feature with search radius 0.050.
:: Downsample with a voxel size 0.010.
:: Estimate normal with search radius 0.020.
:: Compute FPFH feature with search radius 0.050.


In [9]:
# RANSAC algorithm using open3d to estimate R, t 
def register(A_sample, B_sample, A_features, B_features, voxel_size):
    distance_threshold = voxel_size * 1.5
    result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        A_sample,
        B_sample,
        A_features,
        B_features,
        False,
        distance_threshold
    )

    return result.transformation 

T = register(A_sample, B_sample, A_features, B_features, 0.01)
print(T)

[[ 0.80859081 -0.0781031   0.58316447 -0.05051847]
 [ 0.0646157   0.99694294  0.043927   -0.0015447 ]
 [-0.58481254  0.00216261  0.81116559 -0.01331016]
 [ 0.          0.          0.          1.        ]]


# After RANSAC
Let's now apply this transformation matrix to our point cloud data to see how well the point clouds are now registered.

In [12]:
from copy import deepcopy 

A_transformed = deepcopy(cloud_A).transform(T)

widgets.interact(
    plot_point_clouds,
    pcd_list=widgets.fixed([cloud_B, A_transformed]),
    azim=widgets.IntSlider(-90, min=-180, max=180, step=5, description="Azimuth"),
    elev=widgets.IntSlider(90, min=0, max=90, step=5, description="Elevation")
)

interactive(children=(IntSlider(value=-90, description='Azimuth', max=180, min=-180, step=5), IntSlider(value=…

<function utils.plot_point_clouds(pcd_list, azim=-60, elev=30)>

As you can see, `A.transform(T)` and `B: full_bunny` are much better aligned than before but need fine-tuning.

We will now use ICP to try to improve this alignment, a process also called "Local Refinement"

# Basic ICP

In this section, we will implement a basic version of the ICP algorithm. The ICP algorithm has the following steps:



1.   For every point in the source point cloud, find its nearest neighbor in the target point cloud
2.   Find a matrix T that maps points in the source point cloud to the target point cloud while minimizing the euclidean distance between a point and its nearest neighbor (this will be our **error** for each point)
3. Apply this transformation, T, to the source point cloud data
4. Compute the average error for this transformation across all points in the source point cloud data
5. Repeat steps 1 - 4 for N iterations, or break if:

$abs(e_t - e_{t-1}) < \tau$

where $e_i$ is the average error in the i-th iteration and $\tau$ is the tolerance set as a hyperparameter.


In [ ]:
from sklearn.neighbors import NearestNeighbors
from tqdm import trange 

def best_rigid_transform(A, B):
    return T, R, t


widgets.interact(
    plot_point_clouds,
    pcd_list=widgets.fixed([cloud_B, A_transformed_ICP]),
    azim=widgets.IntSlider(-90, min=-180, max=180, step=5, description="Azimuth"),
    elev=widgets.IntSlider(90, min=0, max=90, step=5, description="Elevation")
)

 85%|████████▌ | 17/20 [00:00<00:00, 1400.24it/s]

converged iteration: 17, rmse: 3.059410e-03


interactive(children=(IntSlider(value=-90, description='Azimuth', max=180, min=-180, step=5), IntSlider(value=…

<function utils.plot_point_clouds(pcd_list, azim=-60, elev=30)>

# Robust ICP Using open3D

In [ ]:
threshold = 0.01
max_iterations=3000

print("Apply point-to-point ICP")
point2point_registration = o3d.pipelines.registration.registration_icp(
    cloud_A,
    cloud_B,
    threshold,
    T,
    o3d.pipelines.registration.TransformationEstimationPointToPoint(),
    o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=max_iterations)
)

T_robust = point2point_registration.transformation
print(f"{point2point_registration} \n Transformation: {T_robust}")

A_transformed_robust = deepcopy(cloud_A).transform(T_robust)
widgets.interact(
    plot_point_clouds,
    pcd_list=widgets.fixed([cloud_B, A_transformed_robust]),
    azim=widgets.IntSlider(-90, min=-180, max=180, step=5, description="Azimuth"),
    elev=widgets.IntSlider(90, min=0, max=90, step=5, description="Elevation"))

Apply point-to-point ICP
RegistrationResult with fitness=1.000000e+00, inlier_rmse=5.463349e-04, and correspondence_set size of 40097
Access transformation to get result. 
 Transformation: [[ 8.25884131e-01 -1.04539658e-02  5.63742953e-01 -5.19687649e-02]
 [ 2.58411711e-03  9.99887785e-01  1.47560361e-02 -2.13884047e-04]
 [-5.63833951e-01 -1.07299982e-02  8.25818468e-01 -1.07526263e-02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


interactive(children=(IntSlider(value=-90, description='Azimuth', max=180, min=-180, step=5), IntSlider(value=…

<function utils.plot_point_clouds(pcd_list, azim=-60, elev=30)>